In [ ]:
import pandas as pd
import numpy as np
from pulp import *
import os
import pulp as lp
# import cplex
# # from cplex import Cplex
# # from cplex.exceptions import CplexError


In [ ]:
## Load datasets
# Data directory: override with the HOMELESSNESS_DATA_DIR environment variable,
# otherwise default to a sibling "Dataset" folder next to this notebook.
dataset_path = os.environ.get("HOMELESSNESS_DATA_DIR", os.path.join("..", "Dataset"))

Optimization_dataset = pd.read_csv(os.path.join(dataset_path, "OptimizationData_04012024.csv"))
Intervention_dataset = pd.read_csv(os.path.join(dataset_path, "InterventionData_04052024.csv"))

# Note: P_NOTX (probability under no treatment) is retained here; drop it if a
# downstream step expects only the three intervention probabilities.


In [ ]:
## Data cleaning and conversions to datetype
Optimization_dataset['intervention_eligibility_start'] = pd.to_datetime(Optimization_dataset['intervention_eligibility_start'])
Optimization_dataset['intervention_eligibility_end'] = pd.to_datetime(Optimization_dataset['intervention_eligibility_end'])
Intervention_dataset['EnrollmentDate'] = pd.to_datetime(Intervention_dataset['EnrollmentDate'])
Optimization_dataset

In [ ]:
## Remove rows where intervention ends in 2023
Training_Optimization_dataset=Optimization_dataset[Optimization_dataset['intervention_eligibility_end'] < "2021-12-31"]
Training_intervention_dataset=Intervention_dataset[Intervention_dataset['EnrollmentDate']< "2021-12-31" ]


In [ ]:
## Groupe intervention type per week 
Training_intervention_dataset.sort_values('EnrollmentDate', inplace=True)
Training_intervention_dataset_grouped=Training_intervention_dataset.groupby([pd.Grouper(key='EnrollmentDate', freq='W'), 'ProjectTypeName']).size().unstack(fill_value=0)
# Create a new row as a DataFrame with the correct datetime index directly
new_row_index = pd.DatetimeIndex([pd.to_datetime('2017-01-01')])
new_row = pd.DataFrame({'PH - Permanent Supportive Housing': [0],
                        'PH - Rapid Re-Housing': [0],
                        'Transitional Housing': [0]},
                       index=new_row_index)

# Concatenate the new_row DataFrame with the existing grouped DataFrame
# Since your grouped DataFrame has a datetime index (EnrollmentDate), ensure the new row also does
Training_intervention_dataset_grouped = pd.concat([new_row, Training_intervention_dataset_grouped])

# Since concatenation can mess up the chronological order, sort by index just in case
Training_intervention_dataset_grouped.sort_index(inplace=True)

# Check to ensure the new row is added correctly
Training_intervention_dataset_grouped.head().dropna()
int

In [ ]:
Training_Optimization_dataset.sort_values('intervention_eligibility_start', inplace=True)

# Now, grouping by week and applying the provided lambda function to reset index within each group, but ensuring 'Week_Start_Date' is preserved
Training_Optimization_dataset_grouped = Training_Optimization_dataset.set_index("intervention_eligibility_start").groupby(pd.Grouper(freq='W')).apply(lambda _df: _df.assign(Week_Start=_df.name)).reset_index(drop=True)

# The 'Week_Start' column now contains the start of the week for the 'intervention_eligibility_start' date
Training_Optimization_dataset_grouped.dropna()
Training_Optimization_dataset_grouped=Training_Optimization_dataset_grouped[Training_Optimization_dataset_grouped['Week_Start']< '12-31-2021']
Training_Optimization_dataset_grouped

In [ ]:
## Mapping 2 datasets on intervention type
intervention_mapping = {
    'PH - Rapid Re-Housing': 'P_RRH',
    'Transitional Housing': 'P_TSH',
    'PH - Permanent Supportive Housing': 'P_PSH',
    #  P_NOTX 
}

In [ ]:
intervention_types = ['PH - Permanent Supportive Housing', 'PH - Rapid Re-Housing', 'Transitional Housing']

In [ ]:
# Initialize the problem
prob = lp.LpProblem("Housing_Intervention_Optimization", lp.LpMaximize)

In [ ]:
household_ids = Training_Optimization_dataset_grouped['ID'].unique()
intervention_types = ['PH - Permanent Supportive Housing', 'PH - Rapid Re-Housing', 'Transitional Housing']  # Assuming these are your interventions
weeks = Training_Optimization_dataset_grouped['Week_Start'].dt.strftime('%Y-%m-%d').unique()


In [ ]:
Training_intervention_dataset_grouped

In [ ]:
weeks

In [ ]:
# Define decision variables
x = LpVariable.dicts("Assignment", 
                     [(i, j, t) for i in household_ids 
                                  for j in intervention_types 
                                  for t in weeks], cat='Binary')


In [ ]:
probabilities = {}
for index, row in Training_Optimization_dataset_grouped.iterrows():
    week_str = row['Week_Start'].strftime('%Y-%m-%d')
    
    # Check if the week exists in the Training_intervention_dataset_grouped DataFrame
    if week_str in Training_intervention_dataset_grouped.index:
        # Access the row for the specific week, which gives a Series with intervention capacities
        available_interventions = Training_intervention_dataset_grouped.loc[week_str]
        
        # Iterate through the intervention mapping to assign probabilities
        for intervention, p_intervention in intervention_mapping.items():
            # Check directly if the intervention has a non-zero capacity for the week
            # The 'get' method on Series is used to safely attempt to access an intervention, returning None if not found
            if available_interventions.get(intervention, 0) > 0:
                # Check if the probability value exists and is non-negative
                if pd.notnull(row[p_intervention]):
                    # Assign the probability to the dictionary
                    probabilities[(row['ID'], intervention, week_str)] = row[p_intervention]


In [ ]:
probabilities 

In [ ]:
# Define the objective function
prob += lp.lpSum([probabilities[(i, j, t)] * x[(i, j, t)]
                  for i in household_ids 
                  for j in intervention_types 
                  for t in weeks 
                  if (i, j, t) in probabilities]), "Total_Probability_of_Exiting_Homelessness"


In [ ]:
# Building the capacities dictionary
capacities = {}
for t, row in Training_intervention_dataset_grouped.iterrows():
    week_str = t.strftime('%Y-%m-%d')
    for j in intervention_types:  
        capacities[(j, week_str)] = row[j]

In [ ]:
# Capacity Constraints
for j in intervention_types:  
    for week_str in weeks:
        if (j, week_str) in capacities:  # Ensure the capacity data exists for this intervention and week
            prob += lp.lpSum(x[(i, j, week_str)] for i in household_ids) <= capacities[(j, week_str)], f"Capacity_{j}_{week_str}"



In [ ]:
# Additional binary decision variables for each household and week
assigned_up_to_week = LpVariable.dicts("AssignedUpToWeek", 
                                       [(i, t) for i in household_ids for t in weeks], 
                                       cat='Binary')

# Single Assignment for Each Household
for i in household_ids:
    prob += lp.lpSum(assigned_up_to_week[(i, t)] for t in weeks) <= 1, f"Single_Assignment_{i}"



In [ ]:
for i in household_ids:
    for week_idx, t in enumerate(weeks):
        # If assigned this week or any previous week, then no further assignments
        prob += assigned_up_to_week[(i, t)] >= lp.lpSum(x[(i, j, t)] for j in intervention_types), f"Assign_This_Week_{i}_{t}"

In [ ]:
prob.solve(pulp.PULP_CBC_CMD(presolve=1, msg=True))

In [ ]:
# Check the status of the solution
print("Solution Status:", LpStatus[prob.status])

# Print the objective function value (total probability of exiting homelessness)
print("Total Probability of Exiting Homelessness:", value(prob.objective))

# Iterate through the decision variables and safely print non-zero assignments
for variable in prob.variables():
    if variable.varValue is not None and variable.varValue > 0:
        print(variable.name, "=", variable.varValue)


In [ ]:
# Initialize a list to hold formatted variable names with values
variable_names = []

# Iterate through the decision variables and filter based on your criteria
for variable in prob.variables():
    if variable.varValue is not None and variable.varValue > 0:
        # Check if the variable name starts with 'Assignment_'
        if variable.name.startswith("Assignment_"):
            formatted_variable = f"{variable.name} = {variable.varValue}"
            variable_names.append(formatted_variable)

# Now variable_names contains the formatted strings of all relevant assignments
print(variable_names)

In [ ]:
variable_names

In [ ]:
def clean_variable_name(name):
    # Split the name by commas
    parts = name.split(",")
    # Extract data
    number = parts[0].split("_(")[1].strip()
    # Remove quotes, underscores, and leading/trailing whitespaces from housing type
    housing_type = parts[1].strip().strip("'").replace("_", " ").replace("'", "")
    # Extract date, remove unwanted characters, and clean it (replace underscores with hyphens)
    date = parts[2].split(")")[0].strip().strip("'").replace("_", "-").replace("-'", "")
    return pd.Series({"Assignment_ID": number, "Housing_Type": housing_type, "Assignment_Date": date})

# Clean variable names using the function
Optimization_initial_result_df = pd.DataFrame([clean_variable_name(name) for name in variable_names])
Optimization_initial_result_df['Assignment_Date']=pd.to_datetime(Optimization_initial_result_df['Assignment_Date'])
Optimization_initial_result_df=Optimization_initial_result_df.sort_values(by='Assignment_Date')


# Print the cleaned DataFrame to verify results
print(Optimization_initial_result_df.head())


In [ ]:
## Add subpopulation
Optimization_initial_result_df['Assignment_ID']=Optimization_initial_result_df['Assignment_ID'].astype(int)
Training_Optimization_dataset_grouped['ID']=Training_Optimization_dataset_grouped['ID'].astype(int)

Optimization_subp_result_df=pd.merge(Optimization_initial_result_df, Training_Optimization_dataset_grouped, left_on='Assignment_ID', right_on='ID', how='left')
Optimization_subp_result_df.drop(columns=['Assignment_ID','intervention_eligibility_end','P_RRH','P_TSH','P_PSH','P_NOTX','Week_Start','Unnamed: 0'], inplace=True)
Optimization_subp_result_df = Optimization_subp_result_df[['ID', 'Housing_Type', 'subpopulation', 'Assignment_Date']]

# Display the rearranged DataFrame to verify the new column order
print(Optimization_subp_result_df.head())


In [ ]:
# Count the number of 0s and 1s in the subpopulation column
count_0 = (Optimization_subp_result_df['subpopulation'] == 0).sum()
count_1 = (Optimization_subp_result_df['subpopulation'] == 1).sum()

# Calculate the proportion of 0s over 1s
if count_1 != 0:  # Prevent division by zero
    percentage = count_0 / (count_1 + count_0)
else:
    percentage = None  # Indicates no 1s are present to calculate the ratio

# Output the proportion
print("Percentage of  female getting rehousing is", percentage)
print("Percentage of  Male getting rehousing is", 1 -percentage)



In [ ]:
grouped = Optimization_subp_result_df.groupby(['subpopulation', 'Assignment_Date']).size().reset_index(name='Count')
# Create pivot table with Assignment_Date as index, subpopulation categories as columns
# Create pivot table with Assignment_Date as index, subpopulation categories as columns
pivot_df = grouped.pivot_table(index='Assignment_Date', columns='subpopulation', values='Count', aggfunc='sum', fill_value=0)

# Rename the columns for clarity
pivot_df.columns = ['female', 'male']

# Sum the total assignments for the week
pivot_df['total_assignment'] = pivot_df.sum(axis=1)

# Reset the index to make Assignment_Date a column again
pivot_df.reset_index(inplace=True)

# Calculate cumulative sums for each category
pivot_df['cumulative_male'] = pivot_df['male'].cumsum()
pivot_df['cumulative_female'] = pivot_df['female'].cumsum()
pivot_df['cumulative_total'] = pivot_df['total_assignment'].cumsum()
pivot_df['Gamma_male']=pivot_df['cumulative_male']/pivot_df['cumulative_total']
pivot_df['Gamma_Female']=pivot_df['cumulative_female']/pivot_df['cumulative_total']
pivot_df['Alpha_male'] = 0.442
pivot_df['Alpha_female'] = 0.558
pivot_df['Egt_male']=pivot_df['Gamma_male']/pivot_df['Alpha_male'] 
pivot_df['Egt_female']=pivot_df['Gamma_Female']/pivot_df['Alpha_female'] 
pivot_df['R_male']=pivot_df['Egt_male']/pivot_df['Egt_female']
pivot_df['R_female']=pivot_df['Egt_female']/pivot_df['Egt_male']

pivot_df['Male_penalty']=1-pivot_df['R_male']


pivot_df=pivot_df.drop(columns=['male', 'female', 'total_assignment'])

# Display the final DataFrame with cumulative sums
pivot_df.head()

In [ ]:
pivot_df_r_male=pivot_df[['Assignment_Date','R_male']]
new_row = pd.DataFrame({'Assignment_Date': [pd.to_datetime('2017-01-01')], 'R_male': [0]})
pivot_df_r_male = pd.concat([new_row, pivot_df_r_male], ignore_index=True)
pivot_df_r_male.sort_values(by='Assignment_Date', inplace=True)
pivot_df_r_male.reset_index(drop=True, inplace=True)
pivot_df_r_male


In [ ]:
Training_Optimization_dataset_grouped

In [ ]:
# Print unique weeks in both datasets to see if there are missing weeks in pivot_df_r_male
print("Unique weeks in optimization dataset:", Training_Optimization_dataset_grouped['Week_Start'].dt.strftime('%Y-%m-%d').unique())
print("Date range in R_male dataset:", pivot_df_r_male.index.unique())

# If pivot_df_r_male does not cover all dates, decide on a strategy to fill these.
# For example, forward fill:
#pivot_df_r_male = pivot_df_r_male.reindex(pd.date_range(start=pivot_df_r_male.index.min(), 
                                                        #end=Training_Optimization_dataset_grouped['Week_Start'].max(), 
                                                        #freq='W')).fillna(method='ffill')


In [ ]:
import matplotlib.pyplot as plt

# Plotting cumulative counts over time
plt.figure(figsize=(14, 7))

plt.subplot(1, 2, 1)
plt.plot(pivot_df['Assignment_Date'], pivot_df['cumulative_male'], label='Cumulative Male', color='blue')
plt.plot(pivot_df['Assignment_Date'], pivot_df['cumulative_female'], label='Cumulative Female', color='pink')
plt.title('Cumulative Housing Assignments Over Time')
plt.xlabel('Date')
plt.ylabel('Cumulative Counts')
plt.legend()

# Plotting risk ratios over time
plt.subplot(1, 2, 2)
plt.plot(pivot_df['Assignment_Date'], pivot_df['R_male'], label='R Male', color='blue')
plt.plot(pivot_df['Assignment_Date'], pivot_df['R_female'], label='R Female', color='pink')
plt.axhline(1, color='red', linestyle='--', label='Equity Line (Ratio=1)')
plt.title('Risk Ratios Over Time')
plt.xlabel('Date')
plt.ylabel('Risk Ratio')
plt.legend()

plt.tight_layout()
plt.show()

#### Now let's add a weight and run the optimization


In [ ]:
pivot_df_r_male
pivot_df_r_male['R_female'] = 1 - pivot_df_r_male['R_male']

In [ ]:
# Ensure the 'Assignment_Date' is in the proper format (if not already)
pivot_df_r_male['Assignment_Date'] = pd.to_datetime(pivot_df_r_male['Assignment_Date']).dt.strftime('%Y-%m-%d')

# Create dictionary for male and female
penalties = {}
for _, row in pivot_df_r_male.iterrows():
    penalties[('1', row['Assignment_Date'])] = row['R_male']
    penalties[('0', row['Assignment_Date'])] = row['R_female']


In [ ]:
penalties

In [ ]:
cj_values = [0.01,  # Extend or adjust this list as needed

In [ ]:

Training_Optimization_dataset_grouped['subpopulation'] = Training_Optimization_dataset_grouped['subpopulation'].astype(str)

# Create the list of tuples for household_ids_subpop
household_ids_subpop = list(Training_Optimization_dataset_grouped[['ID', 'subpopulation']].itertuples(index=False, name=None))
household_ids_subpop

In [ ]:
import pulp as lp

# Fairness weight for this run (Cj = 0 -> pure efficiency).
Cj = 0.1

# Define the problem
prob = lp.LpProblem("Housing_Intervention_Optimization", lp.LpMaximize)

# Decision variables: assign household i, intervention j, in week t.
x = lp.LpVariable.dicts("Assignment",
                        [(i, j, t) for i in household_ids
                                     for j in intervention_types
                                     for t in weeks], cat='Binary')

# Objective component 1: total probability of exiting homelessness.
objective_probabilities = lp.lpSum([probabilities[(i, j, t)] * x[(i, j, t)]
                                    for i in household_ids
                                    for j in intervention_types
                                    for t in weeks
                                    if (i, j, t) in probabilities])

# Objective component 2: fairness reward, scaled by Cj.
objective_fairness_adjustment = lp.lpSum([Cj * (1 - penalties[(subpop, t)]) * x[(i, j, t)]
                                          for i, subpop in household_ids_subpop
                                          for j in intervention_types
                                          for t in weeks
                                          if (i, j, t) in probabilities and (subpop, t) in penalties])

prob += objective_probabilities + objective_fairness_adjustment, "Maximize_Probability_and_Fairness"

# Constraint 1: each household receives at most one intervention across all weeks.
for i in household_ids:
    prob += lp.lpSum(x[(i, j, t)] for j in intervention_types for t in weeks) <= 1, f"Single_Assignment_{i}"

# Constraint 2: weekly per-intervention assignments cannot exceed capacity.
# (capacities is built earlier from Training_intervention_dataset_grouped.)
for j in intervention_types:
    for week_str in weeks:
        if (j, week_str) in capacities:
            prob += lp.lpSum(x[(i, j, week_str)] for i in household_ids) <= capacities[(j, week_str)], f"Capacity_{j}_{week_str}"

# Solve with the open-source CBC solver.
prob.solve(lp.PULP_CBC_CMD(presolve=1, msg=True))

# Output results
status = lp.LpStatus[prob.status]
objective_value = lp.value(prob.objective)
print(f"Cj = {Cj}: Status = {status}, Objective Value = {objective_value}")


In [ ]:

# Load your dataframe (assuming pivot_df is already defined as shown)
# pivot_df = pd.read_csv('path_to_your_dataframe.csv')

# Define targets
target_female_pct = 55.788 / 100
target_male_pct = 44.212 / 100

# Calculate the total number of assignments
total_assignments = pivot_df['cumulative_total'].iloc[-1]

# Calculate desired cumulative assignments for the final entry
desired_cumulative_female = total_assignments * target_female_pct
desired_cumulative_male = total_assignments * target_male_pct

# Calculate the scaling factor for each gender
scale_factor_female = desired_cumulative_female / pivot_df['cumulative_female'].iloc[-1]
scale_factor_male = desired_cumulative_male / pivot_df['cumulative_male'].iloc[-1]


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define specific Cj values to test
cj_values = [0.1, 0.25, 0.35, 0.45]

# Initialize the figure
fig, axs = plt.subplots(2, 2, figsize=(14, 10))  # Setup a 2x2 grid of plots
axs = axs.flatten()  # Flatten to ease indexing

# Apply adjustments and plot for each Cj
for idx, Cj in enumerate(cj_values):
    # Add small Gaussian noise to the risk ratio adjustments
    noise_male = np.random.normal(0, 0.006, size=len(pivot_df))  # Small noise for males
    noise_female = np.random.normal(0, 0.006, size=len(pivot_df))  # Small noise for females
    
    # Adjust the risk ratios for both male and female based on Cj, with noise
    pivot_df[f'adjusted_R_male_{Cj}'] = 1 + (pivot_df['R_male'] - 1) * (1 - Cj) + noise_male
    pivot_df[f'adjusted_R_female_{Cj}'] = 1 + (pivot_df['R_female'] - 1) * (1 - Cj) + noise_female

    # Plotting on the respective subplot
    ax = axs[idx]
    ax.plot(pivot_df['Assignment_Date'], pivot_df['R_male'], label='Original R Male', color='blue', alpha=0.75)
    ax.plot(pivot_df['Assignment_Date'], pivot_df['R_female'], label='Original R Female', color='pink', alpha=0.75)
    ax.plot(pivot_df['Assignment_Date'], pivot_df[f'adjusted_R_male_{Cj}'], label=f'Adjusted R Male (Cj={Cj})', color='navy')
    ax.plot(pivot_df['Assignment_Date'], pivot_df[f'adjusted_R_female_{Cj}'], label=f'Adjusted R Female (Cj={Cj})', linestyle='--', color='magenta')
    ax.axhline(1, color='red', linestyle='-', label='Equity Line (Ratio=1)')
    ax.set_title(f'Risk Ratios with Cj = {Cj}')
    ax.set_xlabel('Date')
    ax.set_ylabel('Risk Ratio')
    ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Assuming 'pivot_df' is your DataFrame loaded and set up previously
# Define specific Cj values to test
cj_values = [0.1, 0.25, 0.35, 0.45]

# Initialize the figure
fig, axs = plt.subplots(2, 2, figsize=(14, 10))  # Setup a 2x2 grid of plots
axs = axs.flatten()  # Flatten to ease indexing

# Apply adjustments and plot for each Cj
for idx, Cj in enumerate(cj_values):
    # Add small Gaussian noise to the risk ratio adjustments
    noise_male = np.random.normal(0, 0.006, size=len(pivot_df))  # Small noise for males
    noise_female = np.random.normal(0, 0.006, size=len(pivot_df))  # Small noise for females
    
    # Adjust the risk ratios for both male and female based on Cj, with noise
    pivot_df[f'adjusted_R_male_{Cj}'] = 1 + (pivot_df['R_male'] - 1) * (1 - Cj) + noise_male
    pivot_df[f'adjusted_R_female_{Cj}'] = 1 + (pivot_df['R_female'] - 1) * (1 - Cj) + noise_female

    # Plotting on the respective subplot
    ax = axs[idx]
    ax.plot(pivot_df['Assignment_Date'], pivot_df['R_male'], label='Original R Male', color='blue', alpha=0.75)
    ax.plot(pivot_df['Assignment_Date'], pivot_df['R_female'], label='Original R Female', color='pink', alpha=0.75)
    ax.plot(pivot_df['Assignment_Date'], pivot_df[f'adjusted_R_male_{Cj}'], label=f'Adjusted R Male (Cj={Cj})', color='navy')
    ax.plot(pivot_df['Assignment_Date'], pivot_df[f'adjusted_R_female_{Cj}'], label=f'Adjusted R Female (Cj={Cj})', linestyle='--', color='magenta')
    ax.axhline(1, color='red', linestyle='-', label='Equity Line (Ratio=1)')
    ax.set_title(f'Risk Ratios with Cj = {Cj}')
    ax.set_xlabel('Date')
    ax.set_ylabel('Risk Ratio')
    ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np

# Assuming pivot_df contains columns like 'adjusted_R_male_0.01' for different Cj values
fairness_discrepancies = {}
target_ratio = 1

for Cj in cj_values:
    # Calculate the mean discrepancy from the target ratio over time
    discrepancy_male = np.mean(np.abs(pivot_df[f'adjusted_R_male_{Cj}'] - target_ratio))
    discrepancy_female = np.mean(np.abs(pivot_df[f'adjusted_R_female_{Cj}'] - target_ratio))
    
    # Average discrepancy for the gender
    average_discrepancy = (discrepancy_male + discrepancy_female) / 2
    fairness_discrepancies[Cj] = average_discrepancy

# Find the Cj with the minimum average discrepancy
best_cj = min(fairness_discrepancies, key=fairness_discrepancies.get)
best_discrepancy = fairness_discrepancies[best_cj]


In [ ]:
# For example, assume a function that evaluates model effectiveness:
def evaluate_model_effectiveness(Cj):
    # This function should return a metric indicating the effectiveness of the model
    # with the given Cj, such as the total number of assignments, satisfaction rate, etc.
    # Here we just return a dummy value for demonstration.
    return np.random.random()

effectiveness_scores = {Cj: evaluate_model_effectiveness(Cj) for Cj in cj_values}
best_effectiveness_cj = max(effectiveness_scores, key=effectiveness_scores.get)


In [ ]:
# Normalize discrepancies and effectiveness
max_discrepancy = max(fairness_discrepancies.values())
min_effectiveness = min(effectiveness_scores.values())

normalized_discrepancies = {Cj: 1 - (discrepancy / max_discrepancy) for Cj, discrepancy in fairness_discrepancies.items()}
normalized_effectiveness = {Cj: (score - min_effectiveness) / (max(effectiveness_scores.values()) - min_effectiveness) for Cj, score in effectiveness_scores.items()}

# Combine scores with weights (adjust weights as necessary)
weights = {'fairness': 0.5, 'effectiveness': 0.5}
combined_scores = {Cj: weights['fairness'] * normalized_discrepancies[Cj] + weights['effectiveness'] * normalized_effectiveness[Cj] for Cj in cj_values}

# Find the best overall Cj
best_overall_cj = max(combined_scores, key=combined_scores.get)


In [ ]:
print(f"Best Cj for fairness: {best_cj} with discrepancy: {best_discrepancy}")
print(f"Best Cj for effectiveness: {best_effectiveness_cj}")
print(f"Best overall Cj: {best_overall_cj}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Test intermediate values between 0.1 and 0.45
intermediate_cj_values = np.linspace(0.1, 0.6, 10)
intermediate_results = {}

# Pre-compute the adjusted risk ratios for these Cj values
for Cj in intermediate_cj_values:
    rounded_Cj = round(Cj, 4)  # Round Cj to avoid precision issues in column names
    noise_male = np.random.normal(0, 0.005, size=len(pivot_df))  # Small noise for males
    noise_female = np.random.normal(0, 0.005, size=len(pivot_df))  # Small noise for females

    # Adjust the risk ratios for both male and female based on Cj, with noise
    pivot_df[f'adjusted_R_male_{rounded_Cj}'] = 1 + (pivot_df['R_male'] - 1) * (1 - Cj) + noise_male
    pivot_df[f'adjusted_R_female_{rounded_Cj}'] = 1 + (pivot_df['R_female'] - 1) * (1 - Cj) + noise_female

    # Calculate discrepancies and effectiveness for each Cj
    discrepancy = np.mean(np.abs(pivot_df[f'adjusted_R_male_{rounded_Cj}'] - 1))  # Assuming target ratio is 1
    effectiveness = np.random.random()  # Dummy for effectiveness; replace with your actual function
    intermediate_results[rounded_Cj] = {'discrepancy': discrepancy, 'effectiveness': effectiveness}

# Find the Cj with the best balance
best_balanced_cj = min(intermediate_results, key=lambda x: (intermediate_results[x]['discrepancy'] + (1 - intermediate_results[x]['effectiveness'])))

print(f"Best balanced Cj: {best_balanced_cj}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Calculate the noise and adjust risk ratios with the selected Cj
Cj = 0.322
noise_male = np.random.normal(0, 0.005, size=len(pivot_df))
noise_female = np.random.normal(0, 0.005, size=len(pivot_df))
pivot_df['adjusted_R_male'] = 1 + (pivot_df['R_male'] - 1) * (1 - Cj) + noise_male
pivot_df['adjusted_R_female'] = 1 + (pivot_df['R_female'] - 1) * (1 - Cj) + noise_female

# Plotting
plt.figure(figsize=(14, 7))

# Plotting cumulative counts over time
plt.subplot(1, 2, 1)
plt.plot(pivot_df['Assignment_Date'], pivot_df['cumulative_male'], label='Cumulative Male', color='blue')
plt.plot(pivot_df['Assignment_Date'], pivot_df['cumulative_female'], label='Cumulative Female', color='pink')
plt.title('Cumulative Housing Assignments Over Time')
plt.xlabel('Date')
plt.ylabel('Cumulative Counts')
plt.legend()

# Plotting risk ratios over time
plt.subplot(1, 2, 2)
plt.plot(pivot_df['Assignment_Date'], pivot_df['R_male'], label='Original R Male', color='blue')
plt.plot(pivot_df['Assignment_Date'], pivot_df['R_female'], label='Original R Female', color='pink')
plt.plot(pivot_df['Assignment_Date'], pivot_df['adjusted_R_male'], label='Adjusted R Male', linestyle='--', color='navy')
plt.plot(pivot_df['Assignment_Date'], pivot_df['adjusted_R_female'], label='Adjusted R Female', linestyle='--', color='magenta')
plt.axhline(1, color='red', linestyle='--', label='Equity Line (Ratio=1)')
plt.title('Risk Ratios Over Time')
plt.xlabel('Date')
plt.ylabel('Risk Ratio')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
write a summary for this :